Question:
You are using FAISS for nearest neighbor search, but the following code throws a shape mismatch error when searching. Debug and fix it.

In [ ]:
import faiss
import numpy as np

d = 128
index = faiss.IndexFlatL2(d)

np.random.seed(42)
data = np.random.rand(10, d).astype('float32')
index.add(data)

query = np.random.rand(d).astype('float32')
print(query.shape)
query_2d = query.reshape(1,-1) # Reshaped to 2D format
print(query_2d.shape)

D, I = index.search(query_2d, k=3)
print("Nearest neighbors:", I)


(128,)
(1, 128)
Nearest neighbors: [[7 5 9]]


Question: The following code fails to generate embeddings from the Gemini 1.5 Pro API, throwing an AttributeError. Fix it.

In [ ]:
# Required installation (run once in your environment):
# pip install --upgrade google-genai numpy

from google import genai
import numpy as np

client = genai.Client(api_key=API_Key)  

def get_embedding(text: str) -> np.ndarray:
    response = client.models.embed_content(
        model="gemini-embedding-001",           # Current generally available model (Feb 2026)
        contents=text,                          # Single string is accepted; list[str] also works
    )
    embedding_vector = response.embeddings[0].values
    return np.array(embedding_vector, dtype=np.float32)


# Example usage
text = "GenAI is revolutionizing AI models!"
embedding = get_embedding(text)

print("Embedding shape:", embedding.shape)         
print("Embedding", embedding)


Embedding shape: (3072,)
Embedding [-0.02364815  0.03976178  0.02181967 ... -0.01183254  0.00317258
  0.00149018]


Question: The following Gemini API call is generating poor-quality responses. Identify and fix the temperature and max tokens issue.

In [41]:
from google import genai

client = genai.Client(api_key=API_Key)
prompt = "Explain transformers in AI"

response = client.models.generate_content(
    model = 'gemini-2.5-flash',
    config = {
    'temperature' : 1,
    'max_output_tokens' : 500},
    contents = [prompt]
)


print(response.text)


Transformers are a revolutionary neural network architecture that has transformed the field of Artificial Intelligence, particularly Natural Language


Question:The following Retrieval-Augmented Generation (RAG) model does not retrieve the correct document. Debug the query embedding issue.

In [ ]:
from google import genai
import numpy as np
import faiss
import os

client = genai.Client(api_key=API_Key)  
def get_embedding(text: str, output_dim: int = 768) -> np.ndarray:

    response = client.models.embed_content(
        model="gemini-embedding-001",
        contents=text,
        config=genai.types.EmbedContentConfig(
            output_dimensionality=output_dim
        )
    )
    
    # Extract the vector from the first (and only) embedding
    embedding_vector = response.embeddings[0].values
    
    return np.array(embedding_vector, dtype=np.float32)


d = 768
index = faiss.IndexFlatL2(d)

documents = [
    "AI is transforming industries",
    "FAISS is great for vector search",
    "Generative models are powerful"
]

# Generate embeddings (all will be 768-dimensional)
doc_embeddings = np.array([get_embedding(doc, output_dim=d) for doc in documents])

index.add(doc_embeddings) 
query = "How is AI transforming industries?"
query_vector = get_embedding(query, output_dim=d).reshape(1, -1)

distances, indices = index.search(query_vector, k=1)

retrieved_doc = documents[indices[0][0]]
print("Retrieved Document:", retrieved_doc)


Retrieved Document: AI is transforming industries
